# 03 — Gold: dim_party

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_party` |
| **Grain** | One row per PartyId |
| **Source** | `rpt.vwParty` |
| **PK** | `PartyId` (int) |
| **Rows** | 3,736,109 |

> ⚠️ **Cross-sell critical**: `GlobalPartyId` groups ~3.7M local parties into ~626K global entities.
> Always use `GlobalPartyId` when building cross-sell matrices.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_party"
SOURCE_TABLE = "rpt.vwParty"

print(f"✅ Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

## Cell 2: Read silver source

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"📥 Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Rename `Party` → `PartyName`
- Keep cross-sell critical columns: `GlobalPartyId`, `GlobalPartyRole`, `CompCode`
- Drop: SourceQuery, SourceKey, PartyMappingKey, BusinessKey, BusinessOwnerOrganisationKey/Id,
  EmailAddress, GeographyKey, GeographyId, PhoneNumber, ParentKey, ParentId, SourcePartyId,
  IsObfuscated, LastActivityDate, LastActivityUpdatedDate (CONSTANT), LastUpdatedDate,
  SourceCreatedDate, all ETL dates

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("PartyId").cast("int"),
    F.col("PartyKey").cast("string"),
    F.col("Party").alias("PartyName").cast("string"),
    F.col("GlobalPartyId").cast("int"),
    F.col("GlobalPartyRoleId").cast("int"),
    F.col("GlobalPartyRole").cast("string"),
    F.col("CompCode").cast("string"),
    F.col("DUNSNumber").cast("string"),
    F.col("GlobalCountryId").cast("int"),
    F.col("GlobalCountryCode").cast("string"),
    F.col("DataSourceInstanceId").cast("int"),
    F.col("IsActive").cast("boolean"),
    F.col("IsIndividual").cast("boolean"),
    F.col("PartyRoles").cast("string"),
    F.col("IsDeleted").cast("boolean")
)

print(f"✅ After column select: {df_clean.count():,} rows × {len(df_clean.columns)} cols")
print(f"   Dropped {len(df_src.columns) - len(df_clean.columns)} columns")

## Cell 4: Add "Unknown" member for sentinel value `-1`

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================
unknown_row = spark.createDataFrame([(
    -1,             # PartyId
    "Unknown",      # PartyKey
    "Unknown",      # PartyName
    -1,             # GlobalPartyId
    -1,             # GlobalPartyRoleId
    "Unknown",      # GlobalPartyRole
    None,           # CompCode
    None,           # DUNSNumber
    -1,             # GlobalCountryId
    "Unknown",      # GlobalCountryCode
    -1,             # DataSourceInstanceId
    False,          # IsActive
    False,          # IsIndividual
    None,           # PartyRoles
    False           # IsDeleted
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"✅ Added Unknown member: {df_final.count():,} rows")

## Cell 5: Data quality checks

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("PartyId").distinct().count()
nulls = df_final.filter(F.col("PartyId").isNull()).count()
null_global = df_final.filter(
    (F.col("GlobalPartyId").isNull()) & (F.col("PartyId") != -1)
).count()
has_unknown = df_final.filter(F.col("PartyId") == -1).count()

print(f"✅ DQ Checks")
print(f"   Total rows:         {total:,}")
print(f"   Duplicate PKs:      {dupes}")
print(f"   Null PKs:           {nulls}")
print(f"   Null GlobalPartyId: {null_global:,} (non-unknown rows)")
print(f"   Unknown member:     {'✅' if has_unknown == 1 else '❌'}")

assert dupes == 0, f"❌ Found {dupes} duplicate PartyIds!"
assert nulls == 0, f"❌ Found {nulls} null PartyIds!"
print("\n✅ All DQ checks passed")

## Cell 6: Write to gold lakehouse

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{LAKEHOUSE}.{TABLE}")

print(f"✅ Written: {LAKEHOUSE}.{TABLE}")
print(f"   Rows: {spark.table(f'{LAKEHOUSE}.{TABLE}').count():,}")